In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient IV: do we need a critic?

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position: Part B (policy gradient), session 4 of its 4 sessions**
    - session 2 gave the weight (an advantage) and session 3 the clipped, reusable update: together they are PPO
    - **the limitation we start from:** PPO's advantage needs a learned critic, a second large network to train, store and run, whose mistakes become bias
    - this session fills the third slot: *where can the baseline come from?*
- **The question:** can the baseline come from the episodes themselves, with no learned critic? ("slots" are the three choices of the plan from session 1: weight, how far, baseline source)
- **By the end you can**
    - compute a group-relative advantage from the rewards of a group of completions
    - read the GRPO objective as PPO's clipped ratio with a different advantage
    - say what a group baseline gives up compared with a critic
    - say which models PPO and GRPO each need
    - read the finished slot table for Part B


**Where this session sits in Part B**

<img src="images/partB_arc_b4.png" width=95%>


# The idea: let a group of samples be the baseline

**Why this section?** The baseline only has to be a number that does not depend on the action (session 2). A critic is one source; a second is a group of samples.

- the cost of the critic (from the last session)
    - a second network to train, store and run
    - when the reward arrives only at the end of a long response, the critic must score every prefix
    - its errors bias the advantage
- the alternative: for *one* prompt (for a language model the prompt is the start state), sample a **group** of $G \geq 2$ completions $\tau_1, \ldots, \tau_G$ from the old policy (the policy at the start of the iteration)
    - all start in the same state, so the *average* return of the group estimates the value of that state
    - use that average as the baseline: no learned critic
- this is the session-2 baseline idea with a new source
    - the moving-average baseline of REINFORCE pooled *all* past episodes
    - the group baseline uses the *same prompt*, from this iteration's policy
    - caveat: the group mean contains the completion's own return, so it is not exactly independent of it ("What the group baseline gives up", later in this session, quantifies this)
    - the advantage is "better or worse than my siblings"


## The group-relative advantage

**Why this section?** The one new formula. Everything else in GRPO is session 3.

- completion $i$ has a return $\rlret_i$ (a single number for the whole completion; written $r_i$ in the GRPO papers)
- its advantage is its return relative to the group

$$
\hat{\advseq}_i = \frac{\rlret_i - \operatorname{mean}(\rlret_1, \ldots, \rlret_G)}{\operatorname{std}(\rlret_1, \ldots, \rlret_G)}
$$

- the same advantage $\hat{\advseq}_i$ is used for **every token** of completion $i$
    - the reward is one number per completion, and there is no critic to assign per-step credit (session 2)
- by numbers: one prompt, $G = 4$, returns $(1, 0, 0, 0)$
    - mean $0.25$; sample standard deviation $0.5$ (divide by $G - 1$; the group size $G$ is the papers' symbol, and a bare $G$ is not the return)
    - advantages: $(1 - 0.25)/0.5 = 1.5$ for the success, $(0 - 0.25)/0.5 = -0.5$ for each of the others
- dividing by the standard deviation rescales each prompt's advantages to a common size; it also gives more weight to prompts whose rewards nearly all agree, a known criticism, and implementations add a tiny constant to avoid dividing by zero
- the average over tokens ($1/|\tau_i|$, next slides) likewise shifts weight per token with the length of the completion


**Three prompts, one picture** (synthetic rewards, $G = 4$)

<img src="images/b4_group_advantage.png" width=95%>

- prompt 1: one success among four is a rare event, so it gets a large positive advantage and the failures small negative ones
- prompt 2: two and two: symmetric advantages
- prompt 3: all four succeed, so every advantage is $0$: **no policy-gradient signal from this prompt** (the KL term still acts), and the same happens if all four fail
- the advantage signal comes only from prompts whose completions *differ* in reward


## The GRPO objective: PPO's clipped ratio, a new advantage

**Why this section?** Slot 2 is not new: we keep session 3's clipped ratio. The advantage (slots 3 and 1) is what changes.

$$
\frac{1}{G} \sum_{i=1}^{G} \frac{1}{|\tau_i|} \sum_{\tt} \min{} \bigl( r_{i,\tt}(\theta)\, \hat{\advseq}_i, \;\; \text{clip}\bigl(r_{i,\tt}(\theta),\, 1 - \epsilon_{\text{clip}},\, 1 + \epsilon_{\text{clip}}\bigr)\, \hat{\advseq}_i \bigr) \; - \; \beta \, \KL
$$

- $r_{i,\tt}(\theta)$ is the probability ratio of token $\tt$ of completion $i$, new policy over old (session 3)
- $|\tau_i|$ is the number of tokens of completion $i$: the loss is averaged over tokens, then over completions
- the $\KL$ term, $\KL(\pi_\theta \,\|\, \pi_{\text{ref}})$ estimated on the sampled tokens, is a penalty on drifting from a frozen **reference model** $\pi_{\text{ref}}$ (for a language model, the model we started from), with weight $\beta$
    - it anchors to a different policy than the clipping does (the KL penalty of session 3)
    - a KL to a frozen reference model is *not* part of PPO's original paper (which has a KL to the old policy as an alternative to clipping); RLHF-style PPO often uses it, GRPO puts it in the loss
- notation: $r_{i,\tt}$ is a *ratio* here; the papers also write the completion's return as $r_i$, which we call $\rlret_i$
- the objective is maximized; code minimizes the negative


## GRPO in one loop

**Why this section?** Compare with the PPO loop of session 3: the critic and the GAE are gone, and the group appears.

- **models:** actor $\pi_\theta$, old actor $\pi_{\text{old}}$ (a snapshot), frozen reference $\pi_{\text{ref}}$; no critic
- for each iteration
    - $\pi_{\text{old}} \leftarrow \pi_\theta$ (once per iteration)
    - for each prompt $\x$ in the batch
        - sample a group of $G$ completions from $\pi_{\text{old}}$; one reward $\rew_i$ per completion
        - advantage $\hat{\advseq}_i = (\rew_i - \text{mean}) / (\text{std} + 10^{-8})$, with the mean and standard deviation taken over the $G$ rewards of *this prompt* (standard deviation with divisor $G - 1$; all-equal rewards give advantage $0$); copy $\hat{\advseq}_i$ to every token of completion $i$
        - compute once, with no gradient: $\log \pi_{\text{old}}$ and $\log \pi_{\text{ref}}$ of the sampled tokens
    - for epoch $= 1, \ldots, K$
        - per token: ratio $r = \exp(\log \pi_\theta - \log \pi_{\text{old}})$ (a ratio here, not a reward); $d = \log \pi_{\text{ref}} - \log \pi_\theta$ (only $\pi_\theta$ carries a gradient); KL estimate $e^{d} - d - 1$ (non-negative; it estimates $\KL(\pi_\theta \,\|\, \pi_{\text{ref}})$, exactly so only for the first epoch, when the tokens come from $\pi_\theta$)
        - loss: minus the mean of $\min{}\bigl(r\,\hat{\advseq}, \; \text{clip}(r, 1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}})\,\hat{\advseq}\bigr)$, plus $\beta$ times the mean of the KL estimate (means over the tokens of each completion, then over completions: a design choice, other variants average over all tokens); update $\theta$

- the advantage $A$ of a completion is copied to all of its tokens
- how best to differentiate the KL term is still debated in practice; the estimate above is the common choice


## PPO versus GRPO: what changes

**Why this section?** Slot 2 is unchanged; slot 3 changes, and slot 1 with it.

| | PPO (session 3) | GRPO |
|:--|:--|:--|
| baseline source (slot 3) | learned critic $\statevalfun_\phi$ | mean of a group of completions |
| weight (slot 1) | GAE advantage, one per step | group-relative advantage, one per completion |
| how far (slot 2) | clipped ratio, $K$ epochs | **same** |
| networks to train or hold | actor, critic (and a frozen reference for language models) | actor and a frozen reference |
| extra cost | train the critic | **sample $G$ completions per prompt** |
| per-step credit | yes, through the critic | none: one weight for the whole completion |
| bias in the weight | from the critic's errors | a shrink by $(G-1)/G$ and the scaling by the standard deviation (next slide) |

- the compute moves from a trained critic to generation: a group of $8$ costs about eight times the sampling per prompt, so in practice a batch uses fewer prompts to keep the total fixed
- the old actor is only a snapshot (often just its stored log-probabilities) in both methods


## What the group baseline gives up

**Why this section?** There is no free baseline. Each source trades something, which is the bias and variance thread again.

- **a shrink, plus a distortion:** the group mean includes the completion's own return
    - without the division by the standard deviation, the advantage equals $\frac{G-1}{G}$ times the return minus the mean of the *other* completions: the weight is shrunk by a uniform factor (the factor is $0.5$ for $G = 2$, $0.875$ for $G = 8$), a rescaling and not a reversal; using the other completions' mean (leave-one-out) would remove it
    - dividing by the standard deviation makes each weight depend on all the samples, including its own: a separate, smaller distortion
- **variance:** the baseline is estimated from only $G$ samples (for example $8$), so it is noisy; PPO's critic generalizes across many episodes and states (and has its own bias)
- **no per-step credit:** every token of a completion shares one weight, so a long response with one wrong step punishes all of its tokens (a critic could in principle assign blame by step; in practice critics are often poor at it)
- **no advantage signal when the rewards agree:** all-success or all-failure prompts give zero policy gradient (only the KL term remains), which wastes their samples
- **it fits when rewards are verifiable (a program can check them) and given per completion**, as in the next part; it is a poor fit when we need credit inside a long episode


## The finished slot table, and Part B on one page

**Why this section?** One formula, three slots. This is the map of the whole part.

| Method | Slot 1: weight $\advseq$ | Slot 2: how far | Slot 3: baseline source |
|:--|:--|:--|:--|
| raw theorem (1) | return, no baseline | plain step | none |
| REINFORCE (2) | return minus baseline | plain step | moving average of returns |
| actor-critic (2) | TD error or GAE | plain step | learned critic |
| PPO (3) | GAE with a learned critic | clipped ratio, $K$ epochs per batch | learned critic |
| **GRPO (4)** | group-relative advantage | clipped ratio, $K$ epochs per batch | **mean of a group of completions** |

- all rows share the theorem: a score function $\nabla_\theta \log \pi_\theta$ times a weight, averaged over sampled episodes
- the slots are the only places the methods differ, so a new method is a new choice in one slot
- the slots are separate choices: GRPO keeps PPO's slot 2 and changes slot 3 (and with it slot 1)


## Part B on the threads

**Why this section?** The shared concepts, one line each, for the part we are closing.

- **on-policy versus off-policy:** the theorem is on-policy; the ratio lets us reuse a batch a few epochs
- **bias and variance:** raw return (unbiased, noisy) $\to$ baseline (same mean, less noise) $\to$ critic (less noise, some bias) $\to$ clip (stability for a little bias) $\to$ group (no critic, a shrink and a distortion, a noisy baseline)
- **evaluate versus improve:** the critic evaluates, the actor improves; GRPO drops the evaluator
- **exploration:** the policy is stochastic, so it explores by sampling; a group needs the completions to *differ*
- **backup depth:** GAE and the return-to-go; GRPO uses the whole completion (depth to the end)
- next: the policy is a language model, so Part B's methods are applied there


## Aside: theoretical versus practical REINFORCE

| Feature | Theoretical REINFORCE | Practical (batch, GRPO-like) |
|:--|:--|:--|
| rollouts per prompt | typically 1 | several (for example $8$) |
| baseline | global moving average (historical) | group mean (immediate) |
| variance | very high (small learning rates are usual) | reduced by sampling more |
| update frequency | every trajectory | after a group of trajectories |
| compute per update | low | high (heavy generation) |
| analogy | your lifetime golf average | several swings from the same spot |


# What we established, and what it sets up

- **Established**
    - a baseline only needs to be independent of the action; a **group** of completions for one prompt can supply it
    - the **group-relative advantage** is a completion's return minus the group mean, divided by the group standard deviation, copied to every token
    - **GRPO** is PPO's clipped ratio with that advantage and a KL penalty to a reference model; no critic
    - the price: a noisy, slightly biased baseline, no per-step credit, and no advantage signal when all rewards agree; the cost moves to sampling
    - **Part B in one table:** one theorem, three slots (weight, how far, baseline source)
- **Sets up**
    - **the limitation:** the group baseline needs one reward per completion; where does that reward come from, for a task with no simple checker?
    - next part: the policy is a language model, a response is an episode, and a reward from a program, a model or a judge scores it; GRPO and PPO are applied there
